In [2]:
import pandas as pd

df = pd.read_csv('Данные для тестового задания.csv') 
df['date'] = pd.to_datetime(df['date'])

In [3]:
# 1. MAU (Уникальные пользователи за месяц)
mau = df['user_id'].nunique()
print(f"MAU: {mau}")

MAU: 7639


In [4]:
# 2. DAU (Среднее количество уникальных в день)
dau_mean = df.groupby('date')['user_id'].nunique().mean()
print(f"Средний DAU: {dau_mean}")

Средний DAU: 560.4666666666667


In [8]:
#3. Retention первого дня у пользователей, пришедших в продукт 1 ноября 

first_visit = df.groupby('user_id')['date'].min().reset_index()
first_visit.columns = ['user_id', 'first_date']


df_retention = df.merge(first_visit, on='user_id')


df_retention['days_since_first'] = (df_retention['date'] - df_retention['first_date']).dt.days


cohort_1nov = df_retention[df_retention['first_date'] == '2023-11-01']
total_users = cohort_1nov['user_id'].nunique()
returned_day1 = cohort_1nov[cohort_1nov['days_since_first'] == 1]['user_id'].nunique()

print(f"Retention 1 дня: {(returned_day1 / total_users) * 100:.1f}%")

Retention 1 дня: 26.6%


In [9]:
# 5. Конверсия в просмотр объявления (CR)
# Считаем долю пользователей, у которых view_adverts > 0
users_with_views = df[df['view_adverts'] > 0]['user_id'].nunique()
conversion = (users_with_views / mau) * 100
print(f"Конверсия: {conversion:.2f}%")

Конверсия: 46.31%


In [10]:
# 6. Среднее количество просмотров на одного пользователя
avg_views = df.groupby('user_id')['view_adverts'].sum().mean()
print(f"Среднее кол-во просмотров: {avg_views:.2f}")

Среднее кол-во просмотров: 2.87


In [11]:
#8 AB test

import pandas as pd
from scipy import stats


file_name = 'Данные аб.xlsx'
df = pd.read_excel(file_name)

print("--- АНАЛИЗ А/В-ТЕСТОВ ---")


for i in range(1, 4):
    # Разделяем выручку по группам внутри эксперимента
    test_group = df[(df['experiment_num'] == i) & (df['experiment_group'] == 'test')]['revenue']
    control_group = df[(df['experiment_num'] == i) & (df['experiment_group'] == 'control')]['revenue']
    
    # Считаем ARPU (средняя выручка на пользователя)
    arpu_test = test_group.mean()
    arpu_control = control_group.mean()
    
    # Проводим t-тест Уэлча (он надежнее для разных дисперсий)
    t_stat, p_val = stats.ttest_ind(test_group, control_group, equal_var=False)
    
    # Формируем вывод
    is_significant = p_val < 0.05
    status = "СТАТИСТИЧЕСКИ ЗНАЧИМО" if is_significant else "НЕЗНАЧИМО"
    recommendation = "Внедрять" if (is_significant and arpu_test > arpu_control) else "Не внедрять"
    
    print(f"\nЭксперимент №{i}:")
    print(f"  ARPU Test: {arpu_test:.2f} | ARPU Control: {arpu_control:.2f}")
    print(f"  p-value: {p_val:.4f} ({status})")
    print(f"  Рекомендация: {recommendation}")

--- АНАЛИЗ А/В-ТЕСТОВ ---

Эксперимент №1:
  ARPU Test: 665.74 | ARPU Control: 722.46
  p-value: 0.6890 (НЕЗНАЧИМО)
  Рекомендация: Не внедрять

Эксперимент №2:
  ARPU Test: 332.93 | ARPU Control: 704.65
  p-value: 0.0011 (СТАТИСТИЧЕСКИ ЗНАЧИМО)
  Рекомендация: Не внедрять

Эксперимент №3:
  ARPU Test: 998.67 | ARPU Control: 663.21
  p-value: 0.0603 (НЕЗНАЧИМО)
  Рекомендация: Не внедрять


In [17]:
import pandas as pd


df = pd.read_excel('листеры.xlsx')

# 9. Средний доход на пользователя (ARPU)
# Сначала суммируем выручку для каждого пользователя, затем берем среднее
revenue_per_user = df.groupby('user_id')['revenue'].sum()
mean_revenue = revenue_per_user.mean()

# 10. Медиана возраста пользователя
# Сначала получаем список уникальных пользователей с их возрастом
unique_users_age = df.drop_duplicates(subset='user_id')['age']
median_age = unique_users_age.median()

print(f"Средний доход на пользователя: {mean_revenue:.1f}")
print(f"Медиана возраста: {median_age}")

Средний доход на пользователя: 156.5
Медиана возраста: 28.0
